In [1]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 77.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 80.0 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inc

In [3]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline, AutoTokenizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [5]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [10]:
print("Creating knowledge base")

kb = []

for idx, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print("Knowledge base size:", len(kb))
print(kb[0])

Creating knowledge base
Knowledge base size: 2000
Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.


In [7]:
print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
kb_embeddings = model.encode(
    kb,
    show_progress_bar=True
)

print(kb_embeddings.shape)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

(2000, 384)


In [12]:
index = faiss.IndexFlatL2(
    kb_embeddings.shape[1]
)

index.add(kb_embeddings)

print(index.ntotal)

2000


In [13]:
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ans_150 = str(row_150[row_150["answer"]])

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [14]:
print(prompt_150)
print()
print(labels_150)
print()
print("Correct Answer:", ans_150)

Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.

['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical framework has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on s

In [15]:
result = zs(
    prompt_150,
    candidate_labels=labels_150
)

print(result)

{'sequence': 'Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."', 'The butterfly effect is the phenomenon that a small change in the initial conditio

In [16]:
correct_index = result["labels"].index(ans_150)
correct_score = result["scores"][correct_index]

print("Correct Answer:", ans_150)
print("Probability:", round(correct_score, 3))

Correct Answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability: 0.384


In [17]:
query_embedding = model.encode(
    [prompt_150],
    show_progress_bar=False
)

print(query_embedding.shape)

(1, 384)


In [18]:
D, I = index.search(
    query_embedding,
    k=10
)

print("Retrieved indices:")
print(I)

print("\nDistances:")
print(D)

Retrieved indices:
[[ 663 1701 1269 1532  576  847 1693 1906  168  150]]

Distances:
[[0.26394257 0.26394257 0.2664284  0.2664284  0.2686218  0.26993614
  0.26993614 0.26993614 0.28316098 0.28714597]]


In [19]:
rank = list(I[0]).index(150) + 1

print("Rank of document 150:", rank)

Rank of document 150: 10


In [20]:
wrong_context = kb[999]

print(wrong_context)

A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics.


In [21]:
adversarial_prompt = f"""
Context:
{wrong_context}

Question:
{prompt_150}
"""

print(adversarial_prompt)


Context:
A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics.

Question:
Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.



In [22]:
adv_result = zs(
    adversarial_prompt,
    candidate_labels=labels_150
)

print(adv_result)

{'sequence': '\nContext:\nA thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics.\n\nQuestion:\nSelect the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.\n', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by

In [23]:
correct_index = adv_result["labels"].index(ans_150)

correct_score = adv_result["scores"][correct_index]

print("Probability:", round(correct_score, 3))

Probability: 0.485


In [24]:
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Cross Encoder Loaded Successfully!")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross Encoder Loaded Successfully!


In [25]:
retrieved_indices = I[0]

print(retrieved_indices)

[ 663 1701 1269 1532  576  847 1693 1906  168  150]


In [26]:
docs_10 = [kb[i] for i in retrieved_indices]

print("Top Retrieved Documents:\n")

for i, doc in enumerate(docs_10, 1):
    print(f"Rank {i}:")
    print(doc)
    print("-" * 80)

Top Retrieved Documents:

Rank 1:
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause significant contrasts in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
--------------------------------------------------------------------------------
Rank 2:
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause significant contrasts in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
--------------------------------------------------------------------------------
Rank 3:
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure can cause significant variations in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
--------------------------------------------------------------------------------
Rank 4:
The butterfly effect is the phenome

In [27]:
pairs = [[prompt_150, doc] for doc in docs_10]

print("Number of pairs:", len(pairs))
print("\nFirst pair:\n")
print(pairs[0])

Number of pairs: 10

First pair:

['Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause significant contrasts in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."']


In [28]:
ce_scores = cross_encoder.predict(pairs)

print(ce_scores)

[4.6602316 4.6602316 4.737523  4.737523  4.687043  4.752577  4.752577
 4.752577  4.7072477 4.758512 ]


In [29]:
ranking = sorted(
    zip(retrieved_indices, ce_scores),
    key=lambda x: x[1],
    reverse=True
)

print("Cross-Encoder Ranking:\n")

for rank, (doc_idx, score) in enumerate(ranking, 1):
    print(f"Rank {rank}: Document {doc_idx} | Score = {score:.4f}")

Cross-Encoder Ranking:

Rank 1: Document 150 | Score = 4.7585
Rank 2: Document 847 | Score = 4.7526
Rank 3: Document 1693 | Score = 4.7526
Rank 4: Document 1906 | Score = 4.7526
Rank 5: Document 1269 | Score = 4.7375
Rank 6: Document 1532 | Score = 4.7375
Rank 7: Document 168 | Score = 4.7072
Rank 8: Document 576 | Score = 4.6870
Rank 9: Document 663 | Score = 4.6602
Rank 10: Document 1701 | Score = 4.6602


In [30]:
correct_rank = None

for rank, (doc_idx, score) in enumerate(ranking, 1):
    if doc_idx == 150:
        correct_rank = rank
        break

print("Rank of the correct document:", correct_rank)

Rank of the correct document: 1


In [31]:
row_42 = train.iloc[42]

prompt_42 = str(row_42["prompt"])

print(prompt_42)

Choose the correct answer: What are permutation-inversion groups? based on the given context.


In [32]:
query_embedding = model.encode(
    [prompt_42],
    show_progress_bar=False
)

D, I = index.search(
    query_embedding,
    k=5
)

retrieved_indices = I[0]

print(retrieved_indices)

[1581 1760   42  241  439]


In [33]:
docs_5 = [kb[i] for i in retrieved_indices]

for i, doc in enumerate(docs_5, 1):
    print(f"Document {i}")
    print(doc)
    print("-"*80)

Document 1
Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both.
--------------------------------------------------------------------------------
Document 2
Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both.
--------------------------------------------------------------------------------
Document 3
Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both.
--------------------------------------------------------------------------------
Document 4
Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutati

In [34]:
concatenated_docs = " ".join(docs_5)

print(concatenated_docs[:500])

Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible


In [35]:
rag_string = f"Context: {concatenated_docs} Question: {prompt_42}"

print(rag_string[:500])

Context: Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of both. Permutation-inversion groups are groups of symmetry operations that are energetically


In [36]:
from transformers import AutoTokenizer

bert_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [44]:
tokens = bert_tokenizer(
    rag_string,
    truncation=False
)

In [45]:
token_count = len(tokens["input_ids"])

print("Total Tokens:", token_count)

Total Tokens: 216


In [46]:
true_document = kb[150]

print(true_document)

The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."


In [47]:
rag_prompt = f"""
Context: {true_document}

Question: {prompt_150}
"""

print(rag_prompt)


Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."

Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.



In [48]:
rag_result = zs(
    rag_prompt,
    candidate_labels=labels_150
)

print(rag_result)

{'sequence': '\nContext: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."\n\nQuestion: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.\n', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause 

In [49]:
correct_index = rag_result["labels"].index(ans_150)

correct_score = rag_result["scores"][correct_index]

print("Probability:", round(correct_score, 3))

Probability: 0.976


In [50]:
hits = 0

for idx in range(100):

    # Current row
    row = train.iloc[idx]

    prompt = str(row["prompt"])

    # Correct answer text
    correct_doc = str(row[row["answer"]])

    # Embed the prompt
    query_embedding = model.encode(
        [prompt],
        show_progress_bar=False
    )

    # Retrieve top 5
    D, I = index.search(query_embedding, k=5)

    retrieved_docs = [kb[i] for i in I[0]]

    # Check if the correct answer text appears in any retrieved document
    hit = any(correct_doc in doc for doc in retrieved_docs)

    if hit:
        hits += 1

print("Hits:", hits)

Hits: 73


In [51]:
hit_rate = hits / 100 * 100

print("Hit Rate:", round(hit_rate, 1))

Hit Rate: 73.0


In [52]:
def map_at_3(predictions, correct):

    if correct in predictions:
        rank = predictions.index(correct) + 1
        return 1 / rank

    return 0

In [53]:
scores = []

letters = ["A", "B", "C", "D", "E"]

for idx in range(20):

    row = train.iloc[idx]

    prompt = str(row["prompt"])

    correct_letter = row["answer"]

    candidate_labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    # -------------------------
    # Retrieve
    # -------------------------

    query_embedding = model.encode(
        [prompt],
        show_progress_bar=False
    )

    D, I = index.search(query_embedding, k=5)

    retrieved_docs = [kb[i] for i in I[0]]

    # -------------------------
    # Rerank
    # -------------------------

    pairs = [[prompt, doc] for doc in retrieved_docs]

    ce_scores = cross_encoder.predict(pairs)

    best_document = retrieved_docs[np.argmax(ce_scores)]

    # -------------------------
    # Augment
    # -------------------------

    rag_prompt = f"Context: {best_document} Question: {prompt}"

    # -------------------------
    # Predict
    # -------------------------

    result = zs(
        rag_prompt,
        candidate_labels=candidate_labels
    )

    # -------------------------
    # Convert labels back to A-E
    # -------------------------

    text_to_letter = {
        str(row["A"]): "A",
        str(row["B"]): "B",
        str(row["C"]): "C",
        str(row["D"]): "D",
        str(row["E"]): "E"
    }

    ranked_letters = [
        text_to_letter[label]
        for label in result["labels"]
    ]

    top3 = ranked_letters[:3]

    score = map_at_3(top3, correct_letter)

    scores.append(score)

print(scores)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[1.0, 1.0, 1.0, 1.0, 1.0, 0.5, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


In [54]:
final_map3 = np.mean(scores)

print("Final MAP@3:", round(final_map3, 3))

Final MAP@3: 0.975
